# 1차 제출 재현
HistGradientBoosting (per-group CF + pooled) 블렌딩.

## 1. 라이브러리 · 상수 정의

**무엇을** — 넘파이/판다스와 `HistGradientBoostingRegressor`(히스토그램 기반 부스팅 회귀)를 불러오고, 문제 전반에 쓰이는 상수를 정의합니다.

**왜 / 어떻게**
- `TARGET_COLS` : 예측 대상인 3개 발전 그룹(KPX group 1~3).
- `CAPACITY_KWH` : 각 그룹의 설비용량(kWh). 모델은 절대 발전량이 아니라 **용량계수(CF = 발전량 / 설비용량, 0~1 범위)** 를 학습합니다. 그룹마다 용량이 다르므로 CF로 정규화하면 세 그룹을 같은 스케일에서 다룰 수 있고, 예측 후 용량을 곱해 다시 kWh로 복원합니다.
- `HUB = 117.0` : 풍력 터빈 허브 높이(m). 뒤에서 관측 높이(10 m, 50 m)의 바람을 허브 높이 바람으로 외삽할 때 기준이 됩니다.

In [1]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

D = "../data/"
TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}
HUB = 117.0

## 2. 피처 엔지니어링 함수들

원시 기상예보(LDAPS·GFS)는 여러 격자점(grid)·여러 원시 변수로 이루어져 있습니다. 이를 **예보 시각(`forecast_kst_dtm`)당 1행**의 학습용 피처로 변환하는 함수들입니다. 발전량 예측의 핵심은 "바람이 얼마나 세게 부는가"이므로, 대부분 **풍속을 여러 방식으로 요약**하는 작업입니다.

**각 함수의 역할과 원리**
- `_ldaps_rowfeats` — LDAPS(국지예보)의 격자점별 파생 변수.
  - `ws10`, `ws50` : u·v 바람 성분을 `hypot(u,v)`로 합성한 풍속(피타고라스). 50 m는 max/min 평균으로 대표값을 만듭니다.
  - `ws50_gust` : max−min 변동폭 → **돌풍(gust) 세기** 대리 변수.
  - `alpha` (윈드시어 지수) : 두 높이의 풍속비를 로그로 계산 → 이를 **멱법칙(power law)** `ws_hub = ws50·(117/50)^alpha`에 넣어 허브 높이 풍속을 외삽. 발전에 직접 관련된 높이의 바람을 추정하는 것이 목적입니다.
  - `ws_hub3`, `ws50_3` : **풍력 발전량은 풍속의 세제곱에 비례**(P ∝ v³)하므로 세제곱 항을 명시적 피처로 제공.
  - `rho` : 이상기체 상태식 `p/(R·T)`로 계산한 공기 밀도(발전량은 밀도에도 비례).
- `_gfs_rowfeats` — GFS(전지구예보)의 파생 변수. 80 m·100 m 등 더 높은 고도 바람, 돌풍, 850 hPa 상층풍 등을 확보(LDAPS와 상호보완).
- `_aggregate` — 격자점들을 예보 시각별로 `groupby` 하여 **평균/최대/표준편차**로 압축. 풍속·돌풍 계열만 max/std를 남겨 피처 수를 절제합니다.
- `_wind_direction` — u·v 평균에 `arctan2`를 취해 풍향 각도를 구하고, 각도의 불연속(0°=360°)을 피하려고 **sin/cos로 인코딩**.
- `calendar_features` — 시각/월/연중일을 sin·cos로 변환한 **주기적 시간 피처**(일주기·계절성 반영).
- `_pivot_grid` — 특정 변수(허브풍속 등)를 격자점별 열로 펼쳐, 지점별 바람의 공간 분포를 유지.
- `build_features` — 위 함수들을 호출해 LDAPS·GFS 피처를 모두 조인한 **최종 피처 테이블**(예보 시각 인덱스)을 만듭니다.

In [2]:
def _ldaps_rowfeats(df):
    o = pd.DataFrame(index=df.index)
    u10, v10 = df["heightAboveGround_10_10u"], df["heightAboveGround_10_10v"]
    o["ws10"] = np.hypot(u10, v10)
    u50 = (df["heightAboveGround_50_50MUmax"] + df["heightAboveGround_50_50MUmin"]) / 2
    v50 = (df["heightAboveGround_50_50MVmax"] + df["heightAboveGround_50_50MVmin"]) / 2
    o["ws50"] = np.hypot(u50, v50)
    o["ws50max"] = np.hypot(df["heightAboveGround_50_50MUmax"], df["heightAboveGround_50_50MVmax"])
    o["ws50_gust"] = (df["heightAboveGround_50_50MUmax"] - df["heightAboveGround_50_50MUmin"]).abs() \
        + (df["heightAboveGround_50_50MVmax"] - df["heightAboveGround_50_50MVmin"]).abs()
    o["blws"] = np.hypot(df["heightAboveGround_5_XBLWS"], df["heightAboveGround_5_YBLWS"])
    alpha = np.log(np.clip(o["ws50"], 0.1, None) / np.clip(o["ws10"], 0.1, None)) / np.log(50 / 10)
    alpha = alpha.clip(-0.5, 1.0)
    o["ws_hub"] = o["ws50"] * (HUB / 50.0) ** alpha
    o["ws_hub3"] = o["ws_hub"] ** 3
    o["ws50_3"] = o["ws50"] ** 3
    o["rho"] = df["surface_0_sp"] / (287.05 * (df["heightAboveGround_2_t"]))
    o["t2"] = df["heightAboveGround_2_t"]
    o["sp"] = df["surface_0_sp"]
    o["blh"] = df["etc_0_blh"]
    o["u50"], o["v50"] = u50, v50
    return o


def _gfs_rowfeats(df):
    o = pd.DataFrame(index=df.index)
    o["gws10"] = np.hypot(df["heightAboveGround_10_10u"], df["heightAboveGround_10_10v"])
    o["gws80"] = np.hypot(df["heightAboveGround_80_u"], df["heightAboveGround_80_v"])
    o["gws100"] = np.hypot(df["heightAboveGround_100_100u"], df["heightAboveGround_100_100v"])
    o["gws100_3"] = o["gws100"] ** 3
    o["gust"] = df["surface_0_gust"]
    o["gpbl"] = np.hypot(df["planetaryBoundaryLayer_0_u"], df["planetaryBoundaryLayer_0_v"])
    o["gws850"] = np.hypot(df["isobaricInhPa_850_u"], df["isobaricInhPa_850_v"])
    o["gu100"], o["gv100"] = df["heightAboveGround_100_100u"], df["heightAboveGround_100_100v"]
    return o


def _aggregate(rowfeats, times, prefix):
    rf = rowfeats.copy()
    rf["forecast_kst_dtm"] = times.values
    g = rf.groupby("forecast_kst_dtm")
    agg = g.mean()
    agg.columns = [f"{prefix}_{c}_mean" for c in agg.columns]
    mx = g.max();  mx.columns = [f"{prefix}_{c}_max" for c in mx.columns]
    sd = g.std();  sd.columns = [f"{prefix}_{c}_std" for c in sd.columns]
    keep_mx = [c for c in mx.columns if "ws" in c or "gust" in c]
    keep_sd = [c for c in sd.columns if "ws" in c]
    return pd.concat([agg, mx[keep_mx], sd[keep_sd]], axis=1)


def _wind_direction(rowfeats, times, ucol, vcol, prefix):
    rf = pd.DataFrame({"u": rowfeats[ucol].values, "v": rowfeats[vcol].values,
                       "forecast_kst_dtm": times.values})
    g = rf.groupby("forecast_kst_dtm").mean()
    ang = np.arctan2(g["v"], g["u"])
    return pd.DataFrame({f"{prefix}_wd_sin": np.sin(ang), f"{prefix}_wd_cos": np.cos(ang)}, index=g.index)


def calendar_features(idx):
    dt = pd.to_datetime(pd.Series(idx))
    out = pd.DataFrame(index=idx)
    h, m, doy = dt.dt.hour.values, dt.dt.month.values, dt.dt.dayofyear.values
    out["hour_sin"] = np.sin(2 * np.pi * h / 24);  out["hour_cos"] = np.cos(2 * np.pi * h / 24)
    out["month_sin"] = np.sin(2 * np.pi * m / 12); out["month_cos"] = np.cos(2 * np.pi * m / 12)
    out["doy_sin"] = np.sin(2 * np.pi * doy / 365); out["doy_cos"] = np.cos(2 * np.pi * doy / 365)
    return out


def _pivot_grid(rowfeats, times, gridids, col, prefix):
    rf = pd.DataFrame({"forecast_kst_dtm": times.values, "grid_id": gridids.values, col: rowfeats[col].values})
    p = rf.pivot_table(index="forecast_kst_dtm", columns="grid_id", values=col)
    p.columns = [f"{prefix}_{col}_g{int(g)}" for g in p.columns]
    return p


def build_features(ldaps, gfs):
    ldaps = ldaps.copy(); gfs = gfs.copy()
    lt = pd.to_datetime(ldaps["forecast_kst_dtm"]); gt = pd.to_datetime(gfs["forecast_kst_dtm"])
    lrf = _ldaps_rowfeats(ldaps); grf = _gfs_rowfeats(gfs)
    L = _aggregate(lrf, lt, "l")
    Lwd = _wind_direction(lrf, lt, "u50", "v50", "l")
    G = _aggregate(grf, gt, "g")
    Gwd = _wind_direction(grf, gt, "gu100", "gv100", "g")
    Lg = _pivot_grid(lrf, lt, ldaps["grid_id"], "ws_hub", "l")
    Gg = _pivot_grid(grf, gt, gfs["grid_id"], "gws100", "g")
    feat = (L.join(Lwd, how="left").join(G, how="left").join(Gwd, how="left")
             .join(Lg, how="left").join(Gg, how="left"))
    feat = feat.join(calendar_features(feat.index), how="left")
    return feat

## 3. 데이터 로드 · 피처/타깃 정렬

**무엇을** — 학습·테스트용 원시 기상데이터와 라벨, 제출 양식을 읽고, `build_features`로 피처를 만든 뒤 시각을 키로 정렬합니다.

**왜 / 어떻게**
- 학습·테스트 각각에 대해 `build_features`를 돌려 예보 시각 인덱스의 피처 테이블(`feat`, `feat_te`)을 생성.
- `df = lab.join(feat, how="inner")` : 라벨과 피처를 **시각 기준 inner join** → 피처와 정답이 모두 존재하는 시점만 학습에 사용.
- `Xte` : 제출 양식의 예보 시각에 테스트 피처를 붙여 예측 입력을 구성(`left join`으로 제출 순서 보존). `Xcols`로 학습·예측의 **열 순서를 동일하게** 맞춥니다.

In [3]:
ldaps = pd.read_csv(D+"train/ldaps_train.csv"); gfs = pd.read_csv(D+"train/gfs_train.csv")
lab = pd.read_csv(D+"train/train_labels.csv"); lab["kst_dtm"] = pd.to_datetime(lab["kst_dtm"])
ldaps_te = pd.read_csv(D+"test/ldaps_test.csv"); gfs_te = pd.read_csv(D+"test/gfs_test.csv")
sub = pd.read_csv(D+"sample_submission.csv"); sub["forecast_kst_dtm"] = pd.to_datetime(sub["forecast_kst_dtm"])

feat = build_features(ldaps, gfs); feat.index = pd.to_datetime(feat.index)
feat_te = build_features(ldaps_te, gfs_te); feat_te.index = pd.to_datetime(feat_te.index)
Xcols = list(feat.columns)
df = lab.set_index("kst_dtm").join(feat, how="inner")
Xte = sub.set_index("forecast_kst_dtm").join(feat_te, how="left")[Xcols]

## 4. 모델 학습 — 그룹별(per-group) + 통합(pooled) 두 방식

**무엇을** — 두 갈래로 모델을 학습해 뒤에서 블렌딩할 재료를 만듭니다.

**왜 / 어떻게**
- `make_model` : `loss="absolute_error"`(MAE 최적화 → 이상치에 강건), 낮은 학습률(0.04)·많은 트리(1200)에 **조기종료**로 과적합 방지. 여러 `SEEDS`로 학습해 평균내는 **시드 앙상블**로 분산을 줄입니다.
- **per-group (`per_cf`)** : 그룹마다 별도 모델을 학습. 각 그룹 고유 패턴을 잘 잡지만, 그룹당 데이터가 적으면 불안정.
- **pooled (`pmods`)** : 세 그룹 데이터를 세로로 쌓아(`long_tr`) 한 모델로 학습하되, 어느 그룹인지 알려주는 **원-핫 플래그 `g1/g2/g3`** 를 추가. 데이터가 많아 안정적이고 그룹 간 공통 패턴을 공유.
- 두 방식 모두 타깃은 kWh가 아니라 **용량계수(CF)** 입니다(`/ CAPACITY_KWH`).

In [4]:
GID = {"kpx_group_1": 0, "kpx_group_2": 1, "kpx_group_3": 2}
SEEDS = [42, 7, 2024, 101]
BLEND = {"kpx_group_1": 1.0, "kpx_group_2": 0.7, "kpx_group_3": 0.0}

def make_model(seed):
    return HistGradientBoostingRegressor(loss="absolute_error", learning_rate=0.04, max_iter=1200,
        max_leaf_nodes=63, min_samples_leaf=40, l2_regularization=1.0, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=50, random_state=seed)

per_cf = {}
for tgt in TARGET_COLS:
    tm = df[tgt].notna(); y = df.loc[tm, tgt] / CAPACITY_KWH[tgt]
    ms = [make_model(s).fit(df.loc[tm, Xcols], y) for s in SEEDS]
    per_cf[tgt] = np.mean([m.predict(Xte) for m in ms], axis=0)

Xcols_pool = Xcols + ["g1", "g2", "g3"]; parts = []
for tgt, gid in GID.items():
    s = df.loc[df[tgt].notna(), Xcols].copy()
    s["g1"] = int(gid == 0); s["g2"] = int(gid == 1); s["g3"] = int(gid == 2)
    s["_cf"] = df.loc[s.index, tgt] / CAPACITY_KWH[tgt]; parts.append(s)
long_tr = pd.concat(parts, ignore_index=True)
pmods = [make_model(s).fit(long_tr[Xcols_pool], long_tr["_cf"]) for s in SEEDS]

## 5. 블렌딩 · 예측 · 제출 파일 생성

**무엇을** — per-group과 pooled 예측을 가중 결합해 최종 발전량을 만들고 제출 CSV로 저장합니다.

**왜 / 어떻게**
- 각 그룹에 해당하는 원-핫 플래그를 붙여 pooled 예측(`pool_cf`)을 구합니다.
- `cf = w·per_cf + (1-w)·pool_cf` : `BLEND` 가중치로 두 예측을 혼합. 그룹1은 per-group을 신뢰(w=1.0), 그룹3은 pooled에 전적으로 의존(w=0.0), 그룹2는 절충(0.7). 이는 **그룹별 데이터량·검증 성능에 맞춘 튜닝** 결과입니다.
- `np.clip(cf·용량, 0, 용량)` : CF를 kWh로 되돌리고 **0 ~ 설비용량 범위로 클리핑**(음수·초과 방지).
- 시각을 문자열로 포맷하고 `utf-8-sig`로 저장해 제출 양식에 맞춥니다.

In [5]:
out = sub[["forecast_id", "forecast_kst_dtm"]].copy()
for tgt, gid in GID.items():
    Xg = Xte.copy(); Xg["g1"] = int(gid == 0); Xg["g2"] = int(gid == 1); Xg["g3"] = int(gid == 2)
    pool_cf = np.mean([m.predict(Xg[Xcols_pool]) for m in pmods], axis=0)
    w = BLEND[tgt]; cf = w * per_cf[tgt] + (1 - w) * pool_cf
    out[tgt] = np.clip(cf * CAPACITY_KWH[tgt], 0, CAPACITY_KWH[tgt])

out["forecast_kst_dtm"] = pd.to_datetime(out["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")
out.to_csv(D+"submission_1차.csv", index=False, encoding="utf-8-sig")
out.head()

,forecast_id,forecast_kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,forecast_0001,2025-01-01 01:00:00,18116.328258,18585.644062,16579.922604
1,forecast_0002,2025-01-01 02:00:00,18032.631306,18164.920446,16218.256896
2,forecast_0003,2025-01-01 03:00:00,17187.271289,18235.075905,16307.973884
3,forecast_0004,2025-01-01 04:00:00,17047.894756,18117.386158,16037.521356
4,forecast_0005,2025-01-01 05:00:00,18040.221422,18616.290065,16742.469083


## 6. 재현성 검증

**무엇을 / 왜** — 방금 생성한 예측이 기존에 제출했던 `data/1차.csv`와 완전히 일치하는지 `np.allclose`(허용오차 1e-6)로 확인합니다. 통과하면 이 노트북이 1차 제출 결과를 **결정적으로 재현**함을 보증합니다(리팩터링·환경 변화로 결과가 바뀌지 않았는지 방지하는 안전장치).

In [6]:
ref = pd.read_csv(D+"1차.csv")
assert np.allclose(out[TARGET_COLS].values, ref[TARGET_COLS].values, atol=1e-6)
print("OK: data/1차.csv와 동일")

OK: data/1차.csv와 동일
